In [1]:
import os
import time

from pyspark.sql import SparkSession

os.environ['OBJC_DISABLE_INITIALIZE_FORK_SAFETY'] = 'YES'

try:
    existing_spark = SparkSession.getActiveSession()
    if existing_spark:
        existing_spark.stop()
except:
    pass

for key in list(os.environ.keys()):
    if 'SPARK' in key or 'JAVA_OPTS' in key:
        del os.environ[key]

# --- 2. Cluster Configuration ---
# Format: local-cluster[num_workers, cores_per_worker, memory_per_worker_in_MB]
NUM_EXECUTORS = 2
CORES_PER_EXECUTOR = 6
MEMORY_PER_EXECUTOR_MB = 4096

MASTER_URL = f"local-cluster[{NUM_EXECUTORS}, {CORES_PER_EXECUTOR}, {MEMORY_PER_EXECUTOR_MB}]"

print(f"Running in mode: {MASTER_URL}")

sp_s = (SparkSession.builder
    .master(MASTER_URL)
    .appName("LocalClusterTest")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.executor.cores", "6")
    .config("spark.executor.instances", NUM_EXECUTORS)
    .config("spark.memory.fraction", "0.6")
    .config("spark.sql.shuffle.partitions", "4")  # For tests, less than the default 200
    .getOrCreate()
)

sp_s.sparkContext.setLogLevel("WARN")

# --- 3. Configuration Check ---
print("Session created.")
print(f"Driver Memory Config: {sp_s.conf.get('spark.driver.memory')}")
print(f"Executor Memory Config: {sp_s.conf.get('spark.executor.memory')}")

# Check the number of executors (may take a couple of seconds to start)
time.sleep(3)
num_executors = len(sp_s.sparkContext.parallelize(range(10), NUM_EXECUTORS).glom().collect())
print(f"📊 Active executors (checked via RDD): {num_executors}")

# --- 4. Distribution Test (Example) ---
# To make sure the task went to executors, not stayed on the driver
def print_executor_info(iterator):
    import os
    # Get the executor ID from the process environment variables
    executor_id = os.environ.get('SPARK_EXECUTOR_ID', 'Driver/Local')
    process_id = os.getpid()
    return [f"Executor ID: {executor_id}, PID: {process_id}"]

# Create a dataframe and apply a transformation
df = sp_s.range(0, 10, 1, 4)  # 4 partitions
result = df.rdd.mapPartitions(print_executor_info).collect()

print("\n🖥️ Where tasks were executed:")
for line in result:
    print(line)

sp_s

Running in mode: local-cluster[2, 6, 4096]


26/08/27 14:28:07 WARN Utils: Your hostname, MacBook-Pro-Danil.local resolves to a loopback address: 127.0.0.1; using 192.168.1.64 instead (on interface en0)
26/08/27 14:28:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/27 14:28:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Session created.
Driver Memory Config: 4g
Executor Memory Config: 4g


📊 Active executors (checked via RDD): 2

🖥️ Where tasks were executed:
Executor ID: Driver/Local, PID: 80647
Executor ID: Driver/Local, PID: 80646
Executor ID: Driver/Local, PID: 80656
Executor ID: Driver/Local, PID: 80655


# AB test 

A/B testing is the research method that allows you to find out the effect of a particular change in the product. The study shows which of the two versions of the product or offer gives greater effect on the selected metrics and if it is statistically significant.  

<ul>
  <li><a href="#creation-of-a-new-test-dataset-with-synthetic-data">Creation of a new test dataset with synthetic data.
  <li><a href="#ab-test">AB test.
  <li><a href="#additional-tests-in-ab-test">Additional tests in AB Test.
  <li><a href="#abn-test">ABn Test.
</ul>

In [2]:
import random

from hypex import ABTest
from hypex.dataset import Dataset, InfoRole, TargetRole, TreatmentRole
from hypex.utils import create_test_data, BackendsEnum

/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


## Creation of a new test dataset with synthetic data. 

In order to be able to work with our data in HypEx, first we need to convert it into `dataset`. It is important to mark the data fields by assigning the appropriate `roles`:
- FeatureRole: a role for columns that contain features or predictor variables. Our split will be based on them. Applied by default if the role is not specified for the column.
- TreatmentRole: a role for columns that show the treatment or intervention.
- TargetRole: a role for columns that show the target or outcome variable.
- InfoRole: a role for columns that contain information about the data, such as user IDs. 

In [3]:
df=create_test_data()
df["treat"] = [random.choice([0, 1, 2]) for _ in range(len(df))]
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "treat": TreatmentRole(),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": TargetRole()
    }, 
    data=df,
    session=sp_s,
    backend=BackendsEnum.spark
)
data

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry
0,0.0,0.0,1,462.0,418.666667,37.0,M,Logistics
1,1.0,11.0,2,491.5,438.444444,23.0,M,E-commerce
2,2.0,10.0,0,486.0,456.555556,43.0,F,Logistics
3,3.0,0.0,0,479.5,414.0,47.0,F,E-commerce
4,4.0,0.0,2,477.0,406.555556,26.0,M,Logistics
...,...,...,...,...,...,...,...,...
9995,9995.0,0.0,1,503.0,410.222222,42.0,F,E-commerce
9996,9996.0,0.0,1,463.5,414.666667,44.0,M,E-commerce
9997,9997.0,10.0,1,465.0,444.444444,60.0,M,Logistics
9998,9998.0,0.0,2,477.5,417.777778,62.0,M,Logistics


The roles' data types can be assigned automatically as shown below. Also, the fields, which were not marked, receive Feature role by default.
data["treat"] = [random.choice([0, 1, 2]) for _ in range(len(data))]
data

The roles' data types can be assigned automatically as shown below. Also, the fields, which were not marked, receive Feature role by default.

In [ ]:
data.roles

## AB test
Then we select one of the pre-assembled pipelines, in our case `ABTest`. Also, a custom pipeline can be created based on your specific needs and requirements with custom executors.
After that we wrap our prepared `dataset` into `ExperimentData` to be able to run experiments on it and then execute the test with this data passed as the argument.

In [ ]:
test = ABTest()
result = test.execute(data)

Note: HypEx automatically assumes the smallest value in the `TreatmentRole` column as the control group (typically `0`), and compares each other group (e.g. `1`, `2`) against it. Ensure treatment labels are correctly assigned.


### Experiment results
To show the report with summary of the test we run the `resume` method of the output of the experiment.

It displays the results of the test in the form of a table with the following columns:
- `feature`: name of the target feature, change of which we want to analyze.
- `group`: name of the test group we compare with the control group.
- `TTest pass`: result of the TTest, if it is significant or not.
- `TTest p-value`: p-value of the TTest shows the probability of obtaining the result when the null hypothesis is true. The lower the value the more significant the result is.
- `control mean`: the mean of the feature value across the control group.
- `test mean`: the mean of the feature value across the test group.
- `difference`: the difference between the mean of the test group and the mean of the control group.
- `difference %`: the normalized difference between the mean of the test group and the mean of the control group.

In [ ]:
result.resume

The `TTest pass` column shows whether the difference between groups is statistically significant at the 5% level. 
- `OK` means the difference is significant (p < 0.05).
- `NOT OK` means no significant difference was found.

However, significance does not imply practical importance. Always examine the `difference` and `difference %` columns to assess business relevance.


The method sizes shows the statistics on the groups of the data.

The columns are:
- `control size`: the size of the control group.
- `test size`: the size of the test group.
- `control size %`: the share of the control group in the whole dataset.
- `test size %`: the share of the test group in the whole dataset.
- `group`: name of the test group.

In [ ]:
result.sizes

In [ ]:
result.multitest

### Multiple Testing Correction

When multiple metrics or test groups are analyzed, the chance of false positives increases. The `result.multitest` output shows corrected p-values using Holm's method (default) or Bonferroni if specified. The column `rejected` indicates whether the null hypothesis was rejected after correction.

To change correction method:
```python
test = ABTest(multitest_method="bonferroni")


## Additional tests in AB Test 

It is possible to add u-test and chi2-test in pipeline.

Use `u-test` for numeric variables that are skewed or non-normally distributed. It’s a non-parametric alternative to t-test.

Use `chi2-test` for categorical variables (e.g. gender, conversion rate). Note: t-test is not appropriate for categorical outcomes.

In [4]:
test = ABTest(additional_tests=['t-test', 'u-test', 'chi2-test'])
result = test.execute(data)

/Users/danilsamsutdinov/HypEx/hypex/comparators/abstract.py:258: UserWarning: target_fields_data must have only one column when the comparison is done by groups. 2 passed. pre_spends will be used.
  warnings.warn(


The additional columns are:
- `UTest pass`: result of the UTest, if it is significant or not.
- `UTest p-value`: p-value of the UTest shows the probability of obtaining the result when the null hypothesis is true. The lower the value the more significant the result is.
- `Chi2Test pass`: result of the Chi2Test, if it is significant or not.
- `Chi2Test p-value`: p-value of the Chi2Test shows the probability of obtaining the result when the null hypothesis is true. The lower the value the more significant the result is.

In [5]:
result.resume

,feature,group,control mean,test mean,difference %,difference,StatsTTest pass,StatsTTest p-value,GroupUTest pass,GroupUTest p-value,StatsChi2Test pass,StatsChi2Test p-value
0,"['pre_spends', 'post_spends']",1┆pre_spends,NaN,NaN,NaN,NaN,NOT OK,NaN,NaN,NaN,NaN,NaN
1,"['pre_spends', 'post_spends']",1┆post_spends,NaN,NaN,NaN,NaN,NOT OK,NaN,NaN,NaN,NaN,NaN
2,"['pre_spends', 'post_spends']",2┆pre_spends,NaN,NaN,NaN,NaN,NOT OK,NaN,NaN,NaN,NaN,NaN
3,"['pre_spends', 'post_spends']",2┆post_spends,NaN,NaN,NaN,NaN,NOT OK,NaN,NaN,NaN,NaN,NaN
4,"['pre_spends', 'post_spends']",1,NaN,NaN,NaN,NaN,NaN,NaN,NOT OK,0.389334,NaN,NaN
5,"['pre_spends', 'post_spends']",2,NaN,NaN,NaN,NaN,NaN,NaN,NOT OK,0.562043,NaN,NaN
6,gender,1┆gender,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NOT OK,0.503470
7,gender,2┆gender,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NOT OK,0.819538


26/08/27 18:58:19 WARN HeartbeatReceiver: Removing executor 0 with no recent heartbeats: 1082176 ms exceeds timeout 120000 ms
26/08/27 18:58:19 ERROR TaskSchedulerImpl: Lost executor 0 on 192.168.1.64: Executor heartbeat timed out after 1082176 ms
26/08/27 18:58:19 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_65_8 !
26/08/27 18:58:19 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_65_0 !
26/08/27 18:58:19 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_65_2 !
26/08/27 18:58:19 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_65_10 !
26/08/27 18:58:19 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_65_4 !
26/08/27 18:58:19 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_65_6 !
26/08/27 18:58:19 WARN Master: Removing worker-20260827142807-192.168.1.64-60298 because we got no heartbeat in 60 seconds
26/08/27 18:58:19 ERROR TaskSchedulerImpl: Lost executor 1 on 192.168.

In [ ]:
result.multitest

In [ ]:
result.sizes

## ABn Test 

Finally, we may run multiple ab tests with different methods.

In [ ]:
test = ABTest(multitest_method="bonferroni")
result = test.execute(data)

In [ ]:
result.resume

In [ ]:
result.sizes

In [ ]:
result.multitest

## Advanced Variance Reduction Techniques

For improved statistical power and more sensitive A/B tests, consider using covariate adjustment methods:

### CUPED and CUPAC
**CUPED** (Controlled Experiments Using Pre-Experiment Data) and **CUPAC** (Covariate-Updated Pre-Analysis Correction) are advanced techniques that use historical data to reduce variance in your metrics, allowing you to:

- Detect smaller effects with the same sample size
- Reduce the sample size needed to detect the same effect  
- Increase statistical power of your experiments

These methods work by adjusting your target metrics using correlated historical features that are unaffected by the treatment.

**For a comprehensive guide on implementing these techniques, see the [CUPED & CUPAC Tutorial](СUPED&CUPAC.ipynb).**

Key benefits:
- **CUPED**: Simple single-covariate adjustment using linear regression
- **CUPAC**: Advanced multi-covariate adjustment with flexible model selection (linear, ridge, lasso, CatBoost)/

## Common Pitfalls and Recommendations

- Always assign correct roles: use `TreatmentRole` for group labels, `TargetRole` for outcome metrics. Missing roles may cause incorrect test logic.
- For categorical targets, avoid using `t-test`. Instead, include `chi2-test` in `additional_tests`.
- HypEx does not automatically balance groups. Ensure group sizes are roughly equal and comparable.
- Check for missing values. NaNs may silently affect metric calculation.
- If testing many metrics/groups, interpret results only after multiple testing correction.
- Use `result.sizes` to confirm group balance, and consider A/A testing to verify setup before real A/B.

